# Temporal KG — Source Identity Verification (40-pair audit)

**Purpose:** Validate the timestamp-normalized matches using independent Jira changelog fields and graph Event payloads. This does **not** alter the existing F1 scores. A timestamp match alone is **not** event identity proof. Ambiguous and unavailable evidence are explicitly reported.

**Inputs:** the previously generated `timestamp_identity_review_sample.csv`, original curated Jira tickets and `temporal_kg_v2_output/nodes.jsonl`. Runs on Google Colab; read-only inputs. The review output is a new folder.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, re, collections
import pandas as pd
from IPython.display import display
ROOT=Path('/content/drive/MyDrive')
GC1=ROOT/'GC1'
JIRA=GC1/'GC1_Jira_Dataset'
GRAPH=GC1/'entity_resolution_output'/'temporal_kg_v2_output'
OUT=ROOT/'01_Knowledge_Construction'/'Temporal_KG_Evaluation'/'Results'/'Full_Graph_Deterministic_V2'/'Source_Identity_Verification'
OUT.mkdir(parents=True,exist_ok=True)
SAMPLE_CANDIDATES=[ROOT/'01_Knowledge_Construction'/'Temporal_KG_Evaluation'/'Results'/'Full_Graph_Deterministic_V2'/'Timestamp_Reconciliation'/'timestamp_identity_review_sample.csv', ROOT/'timestamp_identity_review_sample.csv']
SAMPLE=next((p for p in SAMPLE_CANDIDATES if p.is_file()),None)
if SAMPLE is None: raise FileNotFoundError('Upload timestamp_identity_review_sample.csv to MyDrive root or the Timestamp_Reconciliation results folder.')
node_files=list(GRAPH.rglob('nodes.jsonl'))
assert len(node_files)==1, f'Expected one nodes.jsonl under {GRAPH}, found {len(node_files)}'
assert JIRA.is_dir(), f'Missing Jira root: {JIRA}'
print('Sample:',SAMPLE,'Graph:',node_files[0],'Output:',OUT)

Mounted at /content/drive
Sample: /content/drive/MyDrive/01_Knowledge_Construction/Temporal_KG_Evaluation/Results/Full_Graph_Deterministic_V2/Timestamp_Reconciliation/timestamp_identity_review_sample.csv Graph: /content/drive/MyDrive/GC1/entity_resolution_output/temporal_kg_v2_output/nodes.jsonl Output: /content/drive/MyDrive/01_Knowledge_Construction/Temporal_KG_Evaluation/Results/Full_Graph_Deterministic_V2/Source_Identity_Verification


## 1. Load sample and locate relevant source tickets

In [2]:
sample=pd.read_csv(SAMPLE,dtype=str).fillna('')
required={'instance_id','issue_key','event_type','source_time','graph_time'}
assert required.issubset(sample.columns),f'Missing columns: {required-set(sample.columns)}'
sample['source_ts']=pd.to_datetime(sample.source_time,format='mixed',utc=True,errors='coerce')
sample['graph_ts']=pd.to_datetime(sample.graph_time,format='mixed',utc=True,errors='coerce')
assert sample.source_ts.notna().all() and sample.graph_ts.notna().all()
print('Sample rows:',len(sample),'projects:',sample.issue_key.str.split('-').str[0].nunique())
def stream_jsonl(p):
    with open(p,encoding='utf-8-sig') as f:
        for line in f:
            if line.strip():yield json.loads(line)
def norm(x):return re.sub('[^a-z0-9]','',str(x or '').lower())
FIELD_TO_TYPE={'status':'STATUS_CHANGED','assignee':'ASSIGNEE_CHANGED','fixversion':'FIX_VERSION_CHANGED','fixversions':'FIX_VERSION_CHANGED','component':'COMPONENT_CHANGED','components':'COMPONENT_CHANGED','priority':'PRIORITY_CHANGED','duedate':'DUE_DATE_CHANGED','duedatefield':'DUE_DATE_CHANGED'}
DIRECT={'STATUS_CHANGE':'STATUS_CHANGED','ASSIGNEE_CHANGE':'ASSIGNEE_CHANGED','COMPONENT_CHANGE':'COMPONENT_CHANGED','PRIORITY_CHANGE':'PRIORITY_CHANGED','DUE_DATE_CHANGE':'DUE_DATE_CHANGED'}
def aligned_graph_type(p):
    raw=str(p.get('event_type') or '').upper();field=norm((p.get('attributes') or {}).get('jira_field'))
    if raw in DIRECT:return DIRECT[raw]
    if raw=='VERSION_CHANGE' and field in ('fixversion','fixversions'):return 'FIX_VERSION_CHANGED'
    return None
wanted=set(sample.issue_key.str.upper())
projects={x.split('-',1)[0] for x in wanted}
tickets={}
for project in sorted(projects):
    files=list(JIRA.glob(f'*_{project}/jira_tickets.jsonl'))
    if len(files)!=1:
        print('WARNING: missing or ambiguous Jira input',project,len(files));continue
    for ticket in stream_jsonl(files[0]):
        key=str(ticket.get('issue_key') or '').upper()
        if key in wanted:tickets[key]=ticket
print('Source tickets found:',len(tickets),'of',len(wanted))

Sample rows: 40 projects: 13
Source tickets found: 36 of 36


## 2. Build independent changelog evidence and graph event evidence

Matching is restricted to the same instance, issue, normalized event type and respective recorded time. The original Jira changelog supplies `field`, `fromString` and `toString`; the graph supplies `jira_field`, `from_value` and `to_value`. A field/value comparison is a separate identity check.

In [3]:
source_index=collections.defaultdict(list)
for issue,ticket in tickets.items():
    for h in ticket.get('full_changelog') or []:
        ts=pd.to_datetime(h.get('created'),format='mixed',utc=True,errors='coerce')
        if pd.isna(ts):continue
        for item in h.get('items') or []:
            typ=FIELD_TO_TYPE.get(norm(item.get('field')))
            if typ:
                source_index[(issue,typ,ts)].append({'field':item.get('field'),'old':item.get('fromString'),'new':item.get('toString'),'changelog_id':h.get('id')})
graph_index=collections.defaultdict(list)
for node in stream_jsonl(node_files[0]):
    if node.get('label')!='Event':continue
    issue=str(node.get('source_id') or '').upper()
    if issue not in wanted:continue
    try:p=json.loads(node.get('payload_json') or '{}')
    except (ValueError,TypeError):continue
    typ=aligned_graph_type(p)
    if typ is None:continue
    ts=pd.to_datetime(node.get('event_time'),format='mixed',utc=True,errors='coerce')
    if pd.isna(ts):continue
    graph_index[(str(node.get('instance_id') or ''),issue,typ,ts)].append({'uid':node.get('uid'),'field':(p.get('attributes') or {}).get('jira_field'),'old':p.get('from_value'),'new':p.get('to_value'),'payload':p})
print('Indexed source keys:',len(source_index),'graph keys:',len(graph_index))

Indexed source keys: 648 graph keys: 438


## 3. Verify identity without forcing uncertain cases

`CONFIRMED_DETAILS` requires unique source and graph candidates, same normalized field, and both old/new values explicitly present and equal. `FIELD_ONLY` is supporting evidence but **not confirmed**. `AMBIGUOUS` means multiple candidates or inconsistent values. Missing evidence remains `INSUFFICIENT_EVIDENCE`. A clear field/value disagreement is `MISMATCH`. Empty/NULL values do not count as proof of equality.

In [4]:
def norm_value(v):
    if v is None or str(v).strip()=='':return None
    return str(v).strip().casefold()
results=[]
for i,r in sample.iterrows():
    issue=str(r.issue_key).upper();typ=str(r.event_type).upper()
    s=source_index.get((issue,typ,r.source_ts),[])
    g=graph_index.get((r.instance_id,issue,typ,r.graph_ts),[])
    result={'sample_row':i+1,'instance_id':r.instance_id,'issue_key':issue,'event_type':typ,'source_time':r.source_time,'graph_time':r.graph_time,'source_candidates':len(s),'graph_candidates':len(g),'decision':'INSUFFICIENT_EVIDENCE','reason':'Missing source or graph candidate','source_changelog_id':'','graph_uid':'','source_field':'','graph_field':'','source_old':'','graph_old':'','source_new':'','graph_new':''}
    if len(s)>1 or len(g)>1:
        result.update(decision='AMBIGUOUS',reason='Multiple source or graph events share this issue/type/timestamp')
    elif len(s)==1 and len(g)==1:
        a,b=s[0],g[0]
        result.update(source_changelog_id=str(a['changelog_id'] or ''),graph_uid=str(b['uid'] or ''),source_field=str(a['field'] or ''),graph_field=str(b['field'] or ''),source_old=str(a['old'] or ''),graph_old=str(b['old'] or ''),source_new=str(a['new'] or ''),graph_new=str(b['new'] or ''))
        if norm(a['field'])!=norm(b['field']):
            result.update(decision='MISMATCH',reason='Jira changelog field and graph jira_field disagree')
        else:
            comparisons=[(norm_value(a['old']),norm_value(b['old'])),(norm_value(a['new']),norm_value(b['new']))]
            if any(x is not None and y is not None and x!=y for x,y in comparisons):
                result.update(decision='MISMATCH',reason='Conflicting old/new value')
            elif all(x is not None and y is not None and x==y for x,y in comparisons):
                result.update(decision='CONFIRMED_DETAILS',reason='Unique timestamp candidate, same field and both values match')
            else:
                result.update(decision='FIELD_ONLY',reason='Field agrees but old/new values missing or incomplete; manual review needed')
    results.append(result)
review=pd.DataFrame(results)
display(review.decision.value_counts(dropna=False).rename_axis('decision').reset_index(name='count'))
display(review[['issue_key','event_type','decision','reason']].head(20))
review.to_csv(OUT/'identity_verification_40_pairs.csv',index=False)
summary={'sample_rows':len(review),'counts':review.decision.value_counts().to_dict(),'scope':'40 selected candidate pairs; not a population-level identity validation','primary_f1_changed':False,'warning':'Do not treat FIELD_ONLY, AMBIGUOUS or INSUFFICIENT_EVIDENCE as confirmed identity.'}
(OUT/'identity_verification_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print('Saved:',OUT)

,decision,count
0,CONFIRMED_DETAILS,30
1,FIELD_ONLY,8
2,AMBIGUOUS,2


,issue_key,event_type,decision,reason
0,AMBARI-18423,STATUS_CHANGED,CONFIRMED_DETAILS,"Unique timestamp candidate, same field and bot..."
1,MC-125351,STATUS_CHANGED,CONFIRMED_DETAILS,"Unique timestamp candidate, same field and bot..."
2,MC-128243,STATUS_CHANGED,CONFIRMED_DETAILS,"Unique timestamp candidate, same field and bot..."
3,OSSRH-31210,STATUS_CHANGED,CONFIRMED_DETAILS,"Unique timestamp candidate, same field and bot..."
4,SPARK-27300,ASSIGNEE_CHANGED,CONFIRMED_DETAILS,"Unique timestamp candidate, same field and bot..."
5,NEXUS-17280,STATUS_CHANGED,CONFIRMED_DETAILS,"Unique timestamp candidate, same field and bot..."
6,MC-179909,STATUS_CHANGED,CONFIRMED_DETAILS,"Unique timestamp candidate, same field and bot..."
7,JRACLOUD-1688,STATUS_CHANGED,CONFIRMED_DETAILS,"Unique timestamp candidate, same field and bot..."
8,MC-165208,STATUS_CHANGED,CONFIRMED_DETAILS,"Unique timestamp candidate, same field and bot..."
9,CONFSERVER-4063,COMPONENT_CHANGED,FIELD_ONLY,Field agrees but old/new values missing or inc...


Saved: /content/drive/MyDrive/01_Knowledge_Construction/Temporal_KG_Evaluation/Results/Full_Graph_Deterministic_V2/Source_Identity_Verification


## 4. Interpretation
- **CONFIRMED_DETAILS:** strong deterministic support from fields and both transition values (not a substitute for independent human validation).
- **FIELD_ONLY / AMBIGUOUS / INSUFFICIENT_EVIDENCE:** inspect manually using the exported CSV, especially where source or graph fields differ.
- **MISMATCH:** investigate source/graph construction or mapping.

**Do not promote 75.8% to a verified primary F1 based only on this 40-pair convenience sample.** The 44.5% exact baseline and 75.8% ±1s sensitivity result remain unchanged. To validate the full normalized score, apply the confirmed mapping and identity rules to all matched events, with an independently selected review sample and transparent uncertainty reporting.

**Note:** Graph payloads may omit changelog IDs or normalize values. Such cases are intentionally labelled as uncertain, not incorrect.